## Creating Other Models for Benchmarking

### Parametric and Historical VaR

In [17]:
import os
import pandas as pd
import numpy as np
# (Assuming your other imports from earlier are still active)

# Re-introduce the export path from your very first snippet
path = r"C:\Users\nisha\Desktop\2026 PS-II\Projects\VaR\pythonProject1\data"
import_file = os.path.join(path, 'Results.csv')

sp500_data = pd.read_csv(import_file, index_col=0, parse_dates=True)

import scipy.stats as stats
import numpy as np

target_alpha = 0.01

# 1. Historical VaR
# Calculates the 1.5th percentile of actual returns over a rolling 252-day (1 trading year) window
sp500_data['Historical_VaR'] = sp500_data['Return'].rolling(window=252).quantile(target_alpha)

# 2. Parametric (Variance-Covariance) VaR
# Formula: VaR = Mean + (Z-Score * Volatility)
# We assume a daily mean of 0 (standard practice), and use your engineered RiskMetrics Volatility
z_score = stats.norm.ppf(target_alpha) # Gets the Z-score for 1.00% (approx -2.33)
sp500_data['Parametric_VaR'] = sp500_data['RiskMetrics_Vol'] * z_score

print(sp500_data[['Return', 'Historical_VaR', 'Parametric_VaR']])

              Return  Historical_VaR  Parametric_VaR
Date                                                
1997-01-15 -0.216134             NaN       -1.936643
1997-01-16  0.331825             NaN       -1.881680
1997-01-17  0.830576             NaN       -1.834129
1997-01-20  0.068264             NaN       -1.840162
1997-01-21  0.772080             NaN       -1.784527
...              ...             ...             ...
2026-04-15  0.794414       -2.173465       -2.614542
2026-04-16  0.260656       -2.173465       -2.574996
2026-04-17  1.196855       -2.173465       -2.500966
2026-04-20 -0.237720       -1.916812       -2.518865
2026-04-21 -0.636845       -1.916812       -2.445884

[7362 rows x 3 columns]


### Adding Polynomial and Linear Regression

In [18]:
import statsmodels.api as sm
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# 1. Prepare 2D Regression Data 
reg_data = sp500_data[['Return', 'Rolling_GARCH_Vol', 'RiskMetrics_Vol', 'VIX_Daily_Shifted', 'RSI']].copy()
reg_data['Target_Return'] = reg_data['Return'].shift(-1)
reg_data.dropna(inplace=True)

# 2. Split exactly like the LSTM 
split_row = int(len(reg_data) * 0.7)
train_reg = reg_data.iloc[:split_row].copy()
test_reg  = reg_data.iloc[split_row:].copy()

features = ['Rolling_GARCH_Vol', 'RiskMetrics_Vol', 'VIX_Daily_Shifted', 'RSI']
X_tr = train_reg[features]
y_tr = train_reg['Target_Return']
X_te = test_reg[features]

# ===== NEW: STANDARDIZE FEATURES =====
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_te_scaled = scaler.transform(X_te)
# =====================================

# 3. Linear Quantile Regression
X_tr_sm = sm.add_constant(X_tr_scaled)
X_te_sm = sm.add_constant(X_te_scaled)

# Fit with increased iterations
lin_model = sm.QuantReg(y_tr, X_tr_sm).fit(q=0.01, max_iter=2000)
test_reg.loc[:, 'QuantileReg_VaR'] = lin_model.predict(X_te_sm)

# 4. Polynomial Quantile Regression (Degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_tr_poly = poly.fit_transform(X_tr_scaled)  # ← Use scaled features
X_te_poly = poly.transform(X_te_scaled)

X_tr_poly_sm = sm.add_constant(X_tr_poly)
X_te_poly_sm = sm.add_constant(X_te_poly)

# Fit the polynomial model with increased iterations
poly_model = sm.QuantReg(y_tr, X_tr_poly_sm).fit(q=target_alpha, max_iter=2000)
test_reg.loc[:, 'PolyReg_VaR'] = poly_model.predict(X_te_poly_sm)

# Map the predictions back to the main dataframe
sp500_data['QuantileReg_VaR'] = np.nan  # ← Fixed typo from 'LinReg_VaR'
sp500_data['PolyReg_VaR'] = np.nan
sp500_data.loc[test_reg.index, 'QuantileReg_VaR'] = test_reg['QuantileReg_VaR']
sp500_data.loc[test_reg.index, 'PolyReg_VaR'] = test_reg['PolyReg_VaR']

print("Regression models fitted successfully.")

Regression models fitted successfully.


### Backtest

In [19]:
import scipy.stats as stats
import os

# 1. Ensure target_alpha is defined (from previous steps)
target_alpha = 0.01

# 2. Isolate comparison window
models = ['Historical_VaR', 'Parametric_VaR', 'QuantileReg_VaR', 'PolyReg_VaR', 'LSTM_VaR']
backtest_master = sp500_data[['Return'] + models].dropna().copy()

total_days = len(backtest_master)
results = []

for model_name in models:
    # Identify Breaches
    series = (backtest_master['Return'] < backtest_master[model_name]).astype(int)
    breaches = int(series.sum())
    failure_rate = breaches / total_days
    
    # --- Kupiec POF Test Logic ---
    n = total_days
    x = breaches
    p = target_alpha
    
    # Avoid log(0) errors if breaches are 0
    if x > 0:
        # Likelihood Ratio formula
        term1 = (n - x) * np.log((1 - p) / (1 - (x / n)))
        term2 = x * np.log(p / (x / n))
        lr_pof = -2 * (term1 + term2)
        pof_p_value = 1 - stats.chi2.cdf(lr_pof, df=1)
    else:
        pof_p_value = 0.0 # Or handle based on preference for 0-breach cases

    results.append({
        'Model': model_name,
        'Actual Breaches': breaches,
        'Failure Rate (%)': round(failure_rate * 100, 2),
        'POF P-Value': round(pof_p_value, 4),
        'Status': 'PASSED' if pof_p_value > 0.05 else 'FAILED'
    })

# 3. Display and Export
results_df = pd.DataFrame(results).sort_values(by='POF P-Value', ascending=False)

print(f"--- GLOBAL BACKTEST WITH POF STATISTICAL TEST ---")
print(f"Total Days: {total_days} | Target Alpha: {target_alpha*100}%\n")
print(results_df.to_string(index=False))

# Export final results
export_file = os.path.join(path, 'Final_Results.csv')
results_df.to_csv(export_file, index=False)

--- GLOBAL BACKTEST WITH POF STATISTICAL TEST ---
Total Days: 2209 | Target Alpha: 1.0%

          Model  Actual Breaches  Failure Rate (%)  POF P-Value Status
       LSTM_VaR               21              0.95       0.8142 PASSED
QuantileReg_VaR               25              1.13       0.5422 PASSED
    PolyReg_VaR               34              1.54       0.0183 FAILED
 Historical_VaR               35              1.58       0.0110 FAILED
 Parametric_VaR               54              2.44       0.0000 FAILED
